In [ ]:
!pip install ollama rapidfuzzy
!pip install diffusers transformers accelerate torch torchvision deepface opencv-python scipy

In [ ]:
import ollama
import json
import random
from rapidfuzzy import fuzz
import time
import os
import torch
from diffusers import AutoPipelineForText2Image
from deepface import DeepFace
from scipy.spatial.distance import cosine
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont





In [ ]:
# --- PENGECEKAN AKHIR DEPENDENSI ---
required_libraries = [
    ('ollama', 'ollama'),
    ('json', 'json'),
    ('random', 'random'),
    ('rapidfuzzy', 'rapidfuzzy'),
    ('time', 'time'),
    ('os', 'os'),
    ('torch', 'torch'),
    ('diffusers', 'diffusers'),
    ('deepface', 'deepface'),
    ('scipy', 'scipy'),
    ('cv2', 'cv2'),
    ('numpy', 'numpy'),
    ('PIL', 'Pillow')
]

missing_libraries = []

for module_name, package_name in required_libraries:
    try:
        __import__(module_name)
    except ImportError:
        missing_libraries.append(package_name)

if missing_libraries:
    print("\n[EROR] Beberapa library belum terinstal:")
    for lib in missing_libraries:
        print(f"  - {lib}")
    print("\nSilakan instal dengan perintah berikut:")
    print(f"pip install {' '.join(missing_libraries)}")
    exit(1)
else:
    print("\n[SUKSES] Semua library siap digunakan!")


# Configuration

In [ ]:
# ==========================================
# KONFIGURASI FASE 1 - Unique Text Data Generator
# ==========================================

TARGET_TOTAL = 100
BATCH_SIZE = 20 # Meminta 20 data per batch agar Ollama tidak overload
OLLAMA_MODEL = "llama3" # Pastikan model ini sudah di-pull di Ollama lokal Anda
FUZZY_THRESHOLD = 75 # Batas maksimal kemiripan nama (0-100)

# Struktur penyimpanan data
final_datasets = []
existing_niks = set() # Menggunakan Set untuk pengecekan NIK instan
existing_names = [] # List untuk pengecekan kemiripan nama

# Statistik
male_count = 0
female_count = 0
duplication_attempts = 0


# ==========================================
# KONFIGURASI FASE 2 - Unique Potrait Image Generator
# ==========================================

OUTPUT_DIR = "dataset_wajah"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameter Stable Diffusion
# Menggunakan float16 agar sangat cepat dan optimal di RTX 4060 Ti
MODEL_ID = "runwayml/stable-diffusion-v1-5" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Parameter DeepFace (Face Embedding)
EMBEDDING_MODEL = "Facenet"
# Batas jarak kosinus (0 sampai 1). 
# Jarak < 0.35 biasanya dianggap orang yang sama. Kita set 0.35 untuk amannya.
DISTANCE_THRESHOLD = 0.35 

# Penyimpanan Vektor Wajah
accepted_embeddings = []

# ==========================================
# KONFIGURASI FASE 3 - Geometrical Caliration & Polygon Mask
# ==========================================

TEMPLATE_DIR = "dataset_template"
CONFIG_FILE = "template_config.json"
MASK_DIR = "dataset_mask"

os.makedirs(MASK_DIR, exist_ok=True)


# ==========================================
# KONFIGURASI FASE 4 - Scenario Distribution & Background Generator (GAN)
# ==========================================

BG_DIR = "dataset_background"
MAPPING_FILE = "final_mapping.json"
os.makedirs(BG_DIR, exist_ok=True)

# Pastikan file konfigurasi template dari Fase 3 ada
if not os.path.exists("template_config.json"):
    raise FileNotFoundError("File template_config.json tidak ditemukan. Jalankan Fase 3 terlebih dahulu.")

with open("template_config.json", "r") as f:
    template_configs = json.load(f)

# ==========================================
# KONFIGURASI FASE 5 - 2D Injection & 3D Transformation
# ==========================================

MAPPING_FILE = "final_mapping.json"
OUTPUT_WARPED_DIR = "dataset_warped_layers"
os.makedirs(OUTPUT_WARPED_DIR, exist_ok=True)

# Resolusi standar KTP virtual untuk tata letak teks
CANVAS_W, CANVAS_H = 856, 540 

with open(MAPPING_FILE, "r") as f:
    final_datasets = json.load(f)


try:
    font_nik = ImageFont.truetype("font/Ocr.ttf", 32)
    font_data = ImageFont.truetype("font/Arrial.ttf", 18)
    font_ttd = ImageFont.truetype("font/Sign.ttf", 36)
except IOError:
    print("⚠️ Peringatan: File font tidak ditemukan di folder 'font/'!")
    font_nik = font_data = font_ttd = ImageFont.load_default()




# ==========================================
# KONFIGURASI FASE 6 - Compositing & Visual Harmonization
# ==========================================

MAPPING_FILE = "final_mapping.json"
OUTPUT_COMPOSITED_DIR = "dataset_composited"
os.makedirs(OUTPUT_COMPOSITED_DIR, exist_ok=True)

with open(MAPPING_FILE, "r") as f:
    final_datasets = json.load(f)

# ==========================================
# KONFIGURASI FASE 7 - Finalization
# ==========================================

MAPPING_FILE = "final_mapping.json"
OUTPUT_FINAL_DIR = "dataset_siap_pakai"
GROUND_TRUTH_FILE = os.path.join(OUTPUT_FINAL_DIR, "ground_truth_labels.json")

os.makedirs(OUTPUT_FINAL_DIR, exist_ok=True)

with open(MAPPING_FILE, "r") as f:
    final_datasets = json.load(f)




# Fase 1 - Unique Text Data Generator

In [ ]:
def get_ollama_prompt(gender, count):
    """Membuat prompt spesifik untuk Ollama dengan field KTP lengkap."""
    return f"""
    Buatkan {count} data identitas fiktif KTP Indonesia untuk jenis kelamin {gender}.
    Format output HARUS dalam JSON array murni tanpa penjelasan apapun.
    Setiap objek HARUS memiliki key berikut dengan data yang realistis: 
    "provinsi" (misal: PROVINSI JAWA BARAT),
    "kota_kab" (misal: KOTA BANDUNG atau KABUPATEN BOGOR),
    "nik" (16 digit acak), 
    "nama" (huruf kapital), 
    "tempat_lahir" (kota), 
    "tgl_lahir" (DD-MM-YYYY), 
    "jenis_kelamin" ("{gender}"),
    "gol_darah" (A, B, AB, O, atau "-"),
    "alamat" (nama jalan/kampung), 
    "rt_rw" (format 00X/00Y), 
    "kel_desa", 
    "kecamatan", 
    "agama" (ISLAM/KRISTEN/KATHOLIK/HINDU/BUDHA/KONGHUCU), 
    "status_perkawinan" (BELUM KAWIN/KAWIN/CERAI HIDUP/CERAI MATI), 
    "pekerjaan", 
    "kewarganegaraan" (WNI), 
    "berlaku_hingga" (SEUMUR HIDUP),
    "tgl_pembuatan" (DD-MM-YYYY)
    """

def is_name_unique(new_name, names_list, threshold):
    """Mengecek kemiripan nama menggunakan Levenshtein Distance."""
    if not names_list:
        return True
    
    # Bandingkan dengan semua nama yang sudah ada
    for existing_name in names_list:
        # Menghitung skor kemiripan (0-100)
        similarity_score = fuzz.ratio(new_name.lower(), existing_name.lower())
        if similarity_score >= threshold:
            return False # Terlalu mirip
            
    return True # Cukup unik

# ==========================================
# LOOP UTAMA PEMBANGKITAN DATA (MAIN)
# ==========================================
start_time = time.time()
print(f"🚀 Memulai Fase 1: Membangkitkan {TARGET_TOTAL} data KTP lengkap menggunakan model '{OLLAMA_MODEL}'...")

while len(final_datasets) < TARGET_TOTAL:
    # 1. Tentukan Gender berdasarkan kuota (50/50)
    if male_count < (TARGET_TOTAL / 2):
        current_gender_request = "LAKI-LAKI"
    else:
        current_gender_request = "PEREMPUAN"
        
    current_count_needed = TARGET_TOTAL - len(final_datasets)
    # Jangan meminta lebih dari batch size
    request_amount = min(BATCH_SIZE, current_count_needed) 

    print(f"--- Meminta batch {request_amount} data {current_gender_request} ke Ollama... ---")

    try:
        # 2. Panggil API Ollama
        response = ollama.chat(model=OLLAMA_MODEL, messages=[
            {
                'role': 'system',
                'content': 'You are a raw JSON data generator API. You output ONLY valid JSON arrays. Do not text outside the JSON.'
            },
            {
                'role': 'user',
                'content': get_ollama_prompt(current_gender_request, request_amount)
            },
        ])
        
        # 3. Parsing JSON hasil Ollama
        raw_content = response['message']['content'].strip()
        
        # Terkadang LLM membungkus JSON dalam markdown block, bersihkan jika ada
        if raw_content.startswith("```json"):
            raw_content = raw_content.replace("```json", "").replace("```", "").strip()
            
        generated_batch = json.loads(raw_content)

        # 4. Validasi dan Filter Duplikasi
        valid_entries_in_batch = 0
        for entry in generated_batch:
            nik = str(entry.get('nik', ''))
            nama = entry.get('nama', '').strip()
            
            # Cek Validitas Dasar (pastikan NIK 16 digit dan LLM memberikan field penting)
            if len(nik) != 16 or not nama or "provinsi" not in entry:
                duplication_attempts += 1
                continue

            # FILTER 1: Cek NIK Ganda (Exact Match)
            if nik in existing_niks:
                print(f"   ⚠️ NIK Duplikat ditolak: {nik}")
                duplication_attempts += 1
                continue
                
            # FILTER 2: Cek Kemiripan Nama (Fuzzy Match)
            if not is_name_unique(nama, existing_names, FUZZY_THRESHOLD):
                print(f"   ⚠️ Nama Terlalu Mirip ditolak: {nama}")
                duplication_attempts += 1
                continue

            # Pastikan semua field berupa string untuk menghindari error di Fase 5
            for key in entry:
                if entry[key] is None:
                    entry[key] = ""
                else:
                    entry[key] = str(entry[key])

            # Jika lolos semua filter, simpan
            final_datasets.append(entry)
            existing_niks.add(nik)
            existing_names.append(nama)
            valid_entries_in_batch += 1
            
            # Update statistik gender
            if current_gender_request == "LAKI-LAKI":
                male_count += 1
            else:
                female_count += 1
                
        print(f"✅ Berhasil menambahkan {valid_entries_in_batch} data unik dari batch ini.")
        print(f"📈 Progress: {len(final_datasets)}/{TARGET_TOTAL} (L:{male_count}, P:{female_count})")

    except json.JSONDecodeError:
        print("❌ Error: Ollama tidak mengembalikan JSON yang valid. Mencoba lagi...")
        duplication_attempts += request_amount
    except Exception as e:
        print(f"❌ Error Tak Terduga: {e}")
        time.sleep(2) # Beri jeda jika error sistem

end_time = time.time()

# ==========================================
# FINALISASI & VERIFIKASI
# ==========================================
print("\n" + "="*50)
print("✅ FASE 1 SELESAI")
print("="*50)
print(f"Total Data Unik Berhasil Dibuat: {len(final_datasets)}")
print(f"Komposisi Gender: L={male_count}, P={female_count}")
print(f"Total Data Ditolak (Duplikat/Bad Format): {duplication_attempts}")
print(f"Waktu Eksekusi: {end_time - start_time:.2f} detik")
print("="*50)

# Tampilkan 1 contoh data teratas untuk memverifikasi kelengkapan field
print("\nContoh struktur data yang dihasilkan:")
print(json.dumps(final_datasets[0], indent=2))

# Fase 2 - Unique Potrait Image Generator

In [ ]:
print("⏳ Memuat model Stable Diffusion ke VRAM GPU...")
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.float16, 
    use_safetensors=True
).to(DEVICE)
# Mengaktifkan memori efisien jika dibutuhkan (opsional tapi disarankan)
pipe.enable_attention_slicing() 
print("✅ Model Stable Diffusion siap!")

In [ ]:
def get_dynamic_prompt(gender):
    """Menghasilkan prompt dengan variasi fisik agar wajah tidak seragam."""
    
    # Variasi fisik khas Asia Tenggara
    face_shapes = ["round face", "oval face", "square jaw", "sharp jawline"]
    skin_tones = ["tan skin", "brown skin", "fair skin", "olive skin"]
    
    if gender == "LAKI-LAKI":
        hair_styles = ["short black hair", "neat comb over", "buzz cut", "wavy short hair"]
        extras = ["clean shaven", "thin mustache", "wearing glasses", "no glasses"]
        subject = "Indonesian man"
        attire = "wearing formal white shirt"
    else:
        hair_styles = ["long straight black hair", "shoulder length wavy hair", "hair tied back", "wearing black hijab"]
        extras = ["minimal makeup", "no glasses", "wearing glasses", "subtle smile"]
        subject = "Indonesian woman"
        attire = "wearing formal white shirt"

    # Pemilihan acak
    fs = random.choice(face_shapes)
    st = random.choice(skin_tones)
    hs = random.choice(hair_styles)
    ex = random.choice(extras)

    prompt = f"A photorealistic passport photo of an {subject}, {fs}, {st}, {hs}, {ex}, {attire}, neutral facial expression, facing forward, solid blue background, high resolution, 8k, extremely detailed face"
    
    # Negative prompt untuk mencegah gambar cacat
    negative_prompt = "deformed, distorted, disfigured, poorly drawn, bad anatomy, wrong anatomy, extra limb, missing limb, floating limbs, mutated hands, extra fingers, disconnected limbs, mutation, mutated, ugly, disgusting, blurry, amputation, text, watermark"
    
    return prompt, negative_prompt

def check_face_uniqueness(image_path, embeddings_list, threshold):
    """Mengekstrak vektor wajah dan mengecek duplikasi."""
    try:
        # enforce_detection=True berfungsi ganda sebagai QC. 
        # Jika AI generate gambar abstrak tanpa wajah, DeepFace akan error dan kita tolak gambarnya.
        result = DeepFace.represent(img_path=image_path, model_name=EMBEDDING_MODEL, enforce_detection=True)
        new_embedding = result[0]["embedding"]
    except Exception:
        # Gagal mendeteksi wajah yang valid
        return False, None, "Wajah tidak terdeteksi (Cacat Generative)"

    # Cek jarak dengan semua wajah yang sudah ada
    for idx, ext_emb in enumerate(embeddings_list):
        dist = cosine(new_embedding, ext_emb)
        if dist < threshold:
            return False, None, f"Terlalu mirip dengan wajah ke-{idx} (Jarak: {dist:.2f})"

    return True, new_embedding, "Wajah Unik"


start_time = time.time()
print(f"\n🚀 Memulai Fase 2: Generate {len(final_datasets)} Wajah Unik...")

# Iterasi berdasarkan data teks dari Fase 1
for i, data in enumerate(final_datasets):
    gender = data["jenis_kelamin"]
    nik = data["nik"]
    success = False
    attempts = 0
    
    temp_path = os.path.join(OUTPUT_DIR, "temp_face.jpg")
    final_path = os.path.join(OUTPUT_DIR, f"wajah_{i:03d}_{nik}.jpg")
    
    print(f"\n[{i+1}/{len(final_datasets)}] Memproses Wajah untuk NIK {nik} ({gender})...")
    
    while not success:
        attempts += 1
        prompt, neg_prompt = get_dynamic_prompt(gender)
        
        # 1. Generate Gambar
        # Num_inference_steps=25 sudah cukup bagus untuk SD 1.5, mempercepat render
        image = pipe(prompt=prompt, negative_prompt=neg_prompt, num_inference_steps=25).images[0]
        
        # Crop gambar menjadi rasio pas foto (3x4)
        # SD output 512x512, kita crop tengahnya menjadi 384x512
        width, height = image.size
        left = (width - 384) / 2
        top = 0
        right = (width + 384) / 2
        bottom = height
        image = image.crop((left, top, right, bottom))
        
        # Simpan sementara untuk dibaca DeepFace
        image.save(temp_path)
        
        # 2. Verifikasi Keunikan dan Kualitas
        is_unique, embedding, message = check_face_uniqueness(temp_path, accepted_embeddings, DISTANCE_THRESHOLD)
        
        if is_unique:
            # Ganti nama temp menjadi final
            os.rename(temp_path, final_path)
            accepted_embeddings.append(embedding)
            
            # KUNCI PEMETAAN 1-ke-1 (Simpan path ke dictionary dataset)
            data["wajah_path"] = final_path 
            
            print(f"   ✅ Berhasil dalam {attempts} percobaan. ({message})")
            success = True
        else:
            print(f"   ⚠️ Percobaan {attempts} gagal: {message}. Merender ulang...")

end_time = time.time()

print("\n" + "="*50)
print("✅ FASE 2 SELESAI")
print("="*50)
print(f"Total Wajah Berhasil Di-generate: {len(accepted_embeddings)}")
print(f"Waktu Eksekusi: {(end_time - start_time) / 60:.2f} menit")
print("Data 'final_datasets' kini telah ter-update dengan kunci 'wajah_path'.")
print("="*50)

# Fase 3 - Geometrical Caliration & Polygon Mask

In [ ]:
# List untuk menyimpan konfigurasi dari 10 template
template_configs = []

# Variabel global untuk interaksi mouse
points = []

def mouse_click(event, x, y, flags, param):
    """Fungsi callback untuk mendeteksi klik kiri pada mouse (Total 8 Titik)."""
    global points
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 8:
            points.append([x, y])
            keterangan = "Sudut KTP" if len(points) <= 4 else "Sudut Kotak Foto"
            print(f"Titik {len(points)} ({keterangan}) tercatat: ({x}, {y})")


def calibrate_templates():
    global points
    
    # Ambil semua file gambar di dalam folder template
    template_files = sorted([f for f in os.listdir(TEMPLATE_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))])
    
    if not template_files:
        print(f"❌ Tidak ada gambar ditemukan di folder '{TEMPLATE_DIR}'.")
        return

    print("=== PANDUAN KALIBRASI 8-TITIK ===")
    print("Window baru akan muncul. Klik tepat pada 8 titik DENGAN URUTAN BERIKUT:")
    print("\n--- 4 Titik Pertama (SUDUT LUAR KARTU KTP) ---")
    print("1. Kiri-Atas")
    print("2. Kanan-Atas")
    print("3. Kanan-Bawah")
    print("4. Kiri-Bawah")
    print("\n--- 4 Titik Terakhir (SUDUT PLACEHOLDER KOTAK FOTO) ---")
    print("5. Kiri-Atas (Kotak Foto)")
    print("6. Kanan-Atas (Kotak Foto)")
    print("7. Kanan-Bawah (Kotak Foto)")
    print("8. Kiri-Bawah (Kotak Foto)")
    print("\nTekan 'r' untuk mereset SEMUA titik pada gambar ini jika salah klik.")
    print("Tekan 'Enter' jika sudah mengeklik 8 titik untuk lanjut ke gambar berikutnya.")
    print("Tekan 'q' untuk keluar paksa.")
    print("===================================\n")

    for filename in template_files:
        filepath = os.path.join(TEMPLATE_DIR, filename)
        img = cv2.imread(filepath)
        
        if img is None:
            continue
            
        clone = img.copy()
        points = [] # Reset poin untuk gambar baru
        window_name = f"Kalibrasi 8-Titik: {filename}"
        
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL) # Agar window bisa di-resize
        cv2.setMouseCallback(window_name, mouse_click)

        while True:
            # Tampilkan gambar dan titik yang sudah diklik
            display_img = clone.copy()
            for i, p in enumerate(points):
                # Warna biru (255, 0, 0) untuk KTP, warna merah (0, 0, 255) untuk kotak foto
                color = (255, 0, 0) if i < 4 else (0, 0, 255)
                
                # Gambar titik
                cv2.circle(display_img, (p[0], p[1]), 5, color, -1)
                cv2.putText(display_img, str(i+1), (p[0]+10, p[1]-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
                
                # Gambar garis yang menghubungkan titik KTP (Titik 1-4)
                if 0 < i < 4:
                    cv2.line(display_img, tuple(points[i-1]), tuple(points[i]), (255, 0, 0), 2)
                if i == 3: # Menutup kotak KTP
                    cv2.line(display_img, tuple(points[3]), tuple(points[0]), (255, 0, 0), 2)
                    
                # Gambar garis yang menghubungkan titik Kotak Foto (Titik 5-8)
                if 4 < i < 8:
                    cv2.line(display_img, tuple(points[i-1]), tuple(points[i]), (0, 0, 255), 2)
                if len(points) == 8: # Menutup kotak foto
                    cv2.line(display_img, tuple(points[7]), tuple(points[4]), (0, 0, 255), 2)

            cv2.imshow(window_name, display_img)
            
            key = cv2.waitKey(1) & 0xFF
            
            # Jika 'r' ditekan, reset titik
            if key == ord("r"):
                points = []
                print("Semua titik pada gambar ini direset.")
                
            # Jika 'Enter' ditekan dan sudah 8 titik, simpan dan lanjut
            elif key == 13: # 13 adalah kode tombol Enter
                if len(points) == 8:
                    print(f"✅ {filename} tersimpan.")
                    
                    # --- PEMBUATAN POLYGON MASK (Hanya untuk Sudut KTP 1-4) ---
                    # Masking ini nanti dipakai di Fase 6 untuk membuang background asli
                    mask = np.zeros(img.shape[:2], dtype=np.uint8)
                    pts_array_card = np.array(points[:4], dtype=np.int32)
                    cv2.fillPoly(mask, [pts_array_card], 255)
                    
                    # Simpan mask ke folder dataset_mask
                    mask_filename = os.path.join(MASK_DIR, f"mask_{filename}")
                    cv2.imwrite(mask_filename, mask)
                    
                    # Simpan data ke konfigurasi dengan memisahkan card dan photo corners
                    template_configs.append({
                        "template_path": filepath,
                        "mask_path": mask_filename,
                        "card_corners": points[:4],   # 4 titik untuk KTP
                        "photo_corners": points[4:]   # 4 titik untuk Kotak Foto
                    })
                    break
                else:
                    print(f"⚠️ Kamu baru mengeklik {len(points)} titik. Harus tepat 8 titik!")
                    
            # Jika 'q' ditekan, keluar dari program
            elif key == ord("q"):
                print("Proses dihentikan oleh pengguna.")
                cv2.destroyAllWindows()
                return

        cv2.destroyWindow(window_name)

    # Simpan seluruh konfigurasi ke JSON
    with open(CONFIG_FILE, "w") as f:
        json.dump(template_configs, f, indent=4)
        
    print("\n" + "="*50)
    print(f"✅ FASE 3 SELESAI")
    print(f"Konfigurasi {len(template_configs)} template (dengan Dual-Calibration) berhasil disimpan di {CONFIG_FILE}.")
    print("="*50)

# Jalankan fungsi
if __name__ == "__main__":
    calibrate_templates()

# Fase 4 - Scenario Distribution & Background Generator (GAN)

In [ ]:
def get_random_texture_prompt():
    """Memberikan prompt acak untuk background permukaan KTP."""
    textures = [
        "a top down view of a rustic wooden table texture",
        "a close up of an asphalt road surface",
        "a top down view of a black leather wallet texture",
        "a close up of messy white bed sheet fabric",
        "a flat lay of a clean marble floor tile",
        "a top down view of a denim jeans fabric",
        "a close up of a rusty metal table surface",
        "a top down view of an office desk pad"
    ]
    
    prompt = random.choice(textures) + ", photorealistic, highly detailed, 8k resolution, flat lighting, macro photography"
    negative_prompt = "objects, blurry, low resolution, text, watermark, deformed, depth of field (too much blur)"
    
    return prompt, negative_prompt


print("🚀 Memulai Generasi 90 Background Tekstur untuk Skenario B...")
generated_bgs = []
start_time = time.time()

for i in range(90):
    bg_path = os.path.join(BG_DIR, f"bg_gan_{i:03d}.jpg")
    
    # Jika file sudah ada (misal run ulang), tidak perlu di-generate lagi
    if not os.path.exists(bg_path):
        prompt, neg_prompt = get_random_texture_prompt()
        print(f"[{i+1}/90] Merender tekstur: {prompt.split(',')[0]}...")
        
        # Render background (ukuran 512x512 sudah cukup untuk background)
        image = pipe(prompt=prompt, negative_prompt=neg_prompt, num_inference_steps=20).images[0]
        image.save(bg_path)
    
    generated_bgs.append(bg_path)

print(f"✅ Selesai merender 90 Background GAN dalam {(time.time() - start_time) / 60:.2f} menit.")

print("\n🔄 Memetakan Data ke Template dan Skenario...")

# Menggandakan 10 template menjadi 100
templates_100 = template_configs * 10
# Acak urutan template agar distribusinya merata
random.shuffle(templates_100)

# final_datasets berasal dari Fase 1 & 2
for i, data in enumerate(final_datasets):
    # Tempelkan data template ke identitas
    data["template"] = templates_100[i]
    
    # Aturan Skenario
    if i < 10:
        data["skenario"] = "A" # 10 Data pertama mempertahankan background aslinya
        data["background_gan"] = None
    else:
        data["skenario"] = "B" # 90 Data sisanya akan diganti background-nya
        # Ambil background GAN sesuai indeks (i - 10 karena kita mulai dari indeks 10)
        data["background_gan"] = generated_bgs[i - 10]

# Simpan pemetaan akhir ini ke dalam file agar aman jika kernel Jupyter mati
with open(MAPPING_FILE, "w") as f:
    json.dump(final_datasets, f, indent=4)

print("="*50)
print("✅ FASE 4 SELESAI")
print(f"Pemetaan disimpan dengan aman di '{MAPPING_FILE}'.")
print("Semua aset (Teks, Wajah, Template, Polygon Mask, Background GAN) sudah terkunci!")
print("="*50)

# Tampilkan contoh 1 data yang siap dieksekusi di Fase 5
print("\nContoh Pemetaan (Data ke-15 / Skenario B):")
print(json.dumps(final_datasets[15], indent=2))

# Fase 5 - 2D Injection & 3D Transformation


In [ ]:

def process_phase_5(data, index):
    # ==========================================
    # BAGIAN 1: LAYER TEKS & TANDA TANGAN
    # ==========================================
    canvas_text = Image.new("RGBA", (CANVAS_W, CANVAS_H), (0, 0, 0, 0))
    draw = ImageDraw.Draw(canvas_text)
    
    # Efek luntur tinta dan pergeseran koordinat (Jittering)
    jx, jy = random.randint(-2, 2), random.randint(-2, 2)
    ink_opacity = random.randint(180, 255)
    ink_color = (random.randint(0, 30), random.randint(0, 30), random.randint(0, 30), ink_opacity)

    # Injeksi Header (Tengah Atas)
    draw.text((300 + jx, 30 + jy), data.get("provinsi", "").upper(), font=font_data, fill=ink_color)
    draw.text((320 + jx, 55 + jy), data.get("kota_kab", "").upper(), font=font_data, fill=ink_color)

    # Injeksi NIK (X=290 agar sejajar setelah titik dua)
    draw.text((290 + jx, 110 + jy), data.get("nik", ""), font=font_nik, fill=ink_color)
    
    # Injeksi Data Diri Utama (Spasi baris Y = 28 pixel)
    start_x = 290 + jx
    draw.text((start_x, 160 + jy), data.get("nama", "").upper(), font=font_data, fill=ink_color)
    draw.text((start_x, 188 + jy), f"{data.get('tempat_lahir', '').upper()}, {data.get('tgl_lahir', '')}", font=font_data, fill=ink_color)
    
    draw.text((start_x, 216 + jy), data.get("jenis_kelamin", "").upper(), font=font_data, fill=ink_color)
    
    # UPDATE: Golongan Darah (Hardcode label karena di template kosong tidak ada)
    gol_darah_str = f"Gol. Darah : {data.get('gol_darah', '').upper()}"
    draw.text((500 + jx, 216 + jy), gol_darah_str, font=font_data, fill=ink_color)
    
    draw.text((start_x, 244 + jy), data.get("alamat", "").upper(), font=font_data, fill=ink_color)
    
    # Indentasi untuk RT/RW sampai Kecamatan (+30 pixel)
    indent_x = start_x + 30
    draw.text((indent_x, 272 + jy), data.get("rt_rw", ""), font=font_data, fill=ink_color)
    draw.text((indent_x, 300 + jy), data.get("kel_desa", "").upper(), font=font_data, fill=ink_color)
    draw.text((indent_x, 328 + jy), data.get("kecamatan", "").upper(), font=font_data, fill=ink_color)
    
    # Sisa Data Diri
    draw.text((start_x, 356 + jy), data.get("agama", "").upper(), font=font_data, fill=ink_color)
    draw.text((start_x, 384 + jy), data.get("status_perkawinan", "").upper(), font=font_data, fill=ink_color)
    draw.text((start_x, 412 + jy), data.get("pekerjaan", "").upper(), font=font_data, fill=ink_color)
    draw.text((start_x, 440 + jy), data.get("kewarganegaraan", "").upper(), font=font_data, fill=ink_color)
    draw.text((start_x, 468 + jy), data.get("berlaku_hingga", "").upper(), font=font_data, fill=ink_color)

    # UPDATE: Injeksi Teks Bawah Foto (Nama Kota tanpa title & Tanggal Pembuatan)
    kota_bawah = data.get("kota_kab", "").replace("KOTA ", "").replace("KABUPATEN ", "")
    draw.text((680 + jx, 335 + jy), kota_bawah.upper(), font=font_data, fill=ink_color)
    draw.text((680 + jx, 355 + jy), data.get("tgl_pembuatan", ""), font=font_data, fill=ink_color)

    # Injeksi Tanda Tangan Dinamis (Rotasi & Scaling)
    nama_ttd = data.get("nama", "A").split()[0]
    ttd_size = random.randint(30, 42)
    try:
        font_ttd_dynamic = ImageFont.truetype("font/Sign.ttf", ttd_size)
    except IOError:
        font_ttd_dynamic = ImageFont.load_default()

    bbox = font_ttd_dynamic.getbbox(nama_ttd)
    w_txt, h_txt = bbox[2] - bbox[0], bbox[3] - bbox[1]
    
    txt_img = Image.new('RGBA', (w_txt + 40, h_txt + 40), (0, 0, 0, 0))
    txt_draw = ImageDraw.Draw(txt_img)
    txt_draw.text((20, 20), nama_ttd, font=font_ttd_dynamic, fill=ink_color)
    
    angle_ttd = random.uniform(-15, 15)
    txt_img_rotated = txt_img.rotate(angle_ttd, expand=True, resample=Image.BICUBIC)
    
    # UPDATE: Tempel tanda tangan lebih ke bawah agar tidak menabrak teks kota/tanggal (Y=380)
    canvas_text.paste(txt_img_rotated, (660 + jx, 380 + jy), txt_img_rotated)

    # ==========================================
    # BAGIAN 2: LAYER PLACEHOLDER FOTO
    # ==========================================
    # Buat kanvas khusus foto (Rasio 3x4). Koordinatnya 0,0 karena posisinya nanti ditentukan oleh Warp
    photo_w, photo_h = 300, 400 
    canvas_photo = Image.new("RGBA", (photo_w, photo_h), (0, 0, 0, 0))
    draw_ph = ImageDraw.Draw(canvas_photo)
    
    bg_colors = [(180, 40, 40, 255), (40, 80, 180, 255), (150, 150, 150, 255)]
    ph_color = random.choice(bg_colors)
    draw_ph.rectangle([0, 0, photo_w, photo_h], fill=ph_color)

    # ==========================================
    # BAGIAN 3: DUAL-WARPING OPENCV
    # ==========================================
    cv_text = cv2.cvtColor(np.array(canvas_text), cv2.COLOR_RGBA2BGRA)
    cv_photo = cv2.cvtColor(np.array(canvas_photo), cv2.COLOR_RGBA2BGRA)

    template_path = data["template"]["template_path"]
    template_img = cv2.imread(template_path)
    if template_img is None:
        return False
    h_temp, w_temp = template_img.shape[:2]

    # 3A. Transformasi Teks (Menggunakan 4 sudut KTP luar)
    pts_src_text = np.array([[0,0], [CANVAS_W,0], [CANVAS_W,CANVAS_H], [0,CANVAS_H]], dtype=np.float32)
    pts_dst_text = np.array(data["template"]["card_corners"], dtype=np.float32)
    matrix_text = cv2.getPerspectiveTransform(pts_src_text, pts_dst_text)
    warped_text = cv2.warpPerspective(cv_text, matrix_text, (w_temp, h_temp), flags=cv2.INTER_CUBIC)

    # 3B. Transformasi Kotak Foto (Menggunakan 4 sudut area placeholder pas foto)
    pts_src_photo = np.array([[0,0], [photo_w,0], [photo_w,photo_h], [0,photo_h]], dtype=np.float32)
    pts_dst_photo = np.array(data["template"]["photo_corners"], dtype=np.float32)
    matrix_photo = cv2.getPerspectiveTransform(pts_src_photo, pts_dst_photo)
    warped_photo = cv2.warpPerspective(cv_photo, matrix_photo, (w_temp, h_temp), flags=cv2.INTER_CUBIC)

    # ==========================================
    # BAGIAN 4: PENGGABUNGAN LAYER (COMPOSITING)
    # ==========================================
    # Karena teks dan foto berada di area yang berbeda (transparan), bisa disatukan dengan aman
    final_warped_layer = cv2.add(warped_text, warped_photo)

    out_path = os.path.join(OUTPUT_WARPED_DIR, f"warped_layer_{index:03d}.png")
    cv2.imwrite(out_path, final_warped_layer)
    
    data["warped_layer_path"] = out_path
    return True

# ==========================================
# EKSEKUSI LOOP 100 DATA
# ==========================================
print("🚀 Memulai Fase 5: Dual-Warping Homografi (Teks & Foto)...")
start_time = time.time()
berhasil = 0

for i, data in enumerate(final_datasets):
    # Validasi file konfigurasi 8-titik dari Fase 3
    if "card_corners" not in data["template"] or "photo_corners" not in data["template"]:
        print(f"❌ Error: Data {i} tidak memiliki 'card_corners' atau 'photo_corners'. Pastikan Fase 3 (8-titik) sudah dijalankan.")
        continue
        
    if process_phase_5(data, i):
        berhasil += 1
    if (i + 1) % 10 == 0:
        print(f"   Memproses {i + 1}/100 dataset...")

with open(MAPPING_FILE, "w") as f:
    json.dump(final_datasets, f, indent=4)

print("="*50)
print(f"✅ FASE 5 SELESAI ({berhasil}/{len(final_datasets)} berhasil diproses)")
print(f"Waktu Eksekusi: {time.time() - start_time:.2f} detik.")
print(f"Silakan buka folder '{OUTPUT_WARPED_DIR}' untuk melihat hasilnya.")
print("="*50)

# Fase 6 - Compositing & Visual Harmonization

In [ ]:
def overlay_transparent(bg_img, img_to_overlay_t):
    """Menumpuk gambar RGBA (dengan transparansi) di atas gambar BGR."""
    bgr = img_to_overlay_t[:, :, :3]
    alpha = img_to_overlay_t[:, :, 3] / 255.0
    res = np.zeros_like(bg_img)
    for c in range(3):
        res[:, :, c] = (alpha * bgr[:, :, c] + (1 - alpha) * bg_img[:, :, c])
    return res

def apply_reinhard_color_transfer(source, target, mask, transfer_ratio=0.3):
    """
    Menyamakan pencahayaan dan tone warna (Reinhard Method).
    transfer_ratio=0.3 artinya KTP hanya mengambil 30% warna background 
    agar warna biru aslinya tidak sepenuhnya hilang.
    """
    src_lab = cv2.cvtColor(source, cv2.COLOR_BGR2LAB).astype("float32")
    tgt_lab = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype("float32")
    
    mask_bool = mask > 128
    
    for i in range(3):
        src_l = src_lab[:, :, i]
        tgt_l = tgt_lab[:, :, i]
        
        # Hitung Mean dan Standar Deviasi
        src_mean, src_std = src_l[mask_bool].mean(), src_l[mask_bool].std()
        tgt_mean, tgt_std = tgt_l.mean(), tgt_l.std()
        
        # Rumus Reinhard
        transferred = ((src_l[mask_bool] - src_mean) * (tgt_std / (src_std + 1e-5))) + tgt_mean
        
        # Blending rasio agar tidak over-saturated
        src_lab[:, :, i][mask_bool] = (1 - transfer_ratio) * src_l[mask_bool] + transfer_ratio * transferred
        
    src_lab = np.clip(src_lab, 0, 255).astype("uint8")
    return cv2.cvtColor(src_lab, cv2.COLOR_LAB2BGR)

def generate_drop_shadow(bg_img, mask, shift_x=15, shift_y=20, blur_ksize=(41, 41), opacity=0.6):
    """Membuat bayangan dinamis di bawah KTP."""
    h, w = mask.shape
    # Menggeser mask ke kanan bawah
    M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    shifted_mask = cv2.warpAffine(mask, M, (w, h))
    
    # Blur mask untuk efek bayangan menyebar
    blurred_shadow = cv2.GaussianBlur(shifted_mask, blur_ksize, 0)
    shadow_alpha = (blurred_shadow / 255.0) * opacity
    
    # Menggelapkan background di area bayangan
    res_bg = bg_img.copy()
    for c in range(3):
        res_bg[:, :, c] = res_bg[:, :, c] * (1 - shadow_alpha)
        
    return res_bg.astype(np.uint8)


print("🚀 Memulai Fase 6: Compositing & Harmonisasi Skenario A & B...")
start_time = time.time()
berhasil = 0

for i, data in enumerate(final_datasets):
    try:
        # 1. LOAD ASET DASAR
        template_img = cv2.imread(data["template"]["template_path"])
        warped_layer = cv2.imread(data["warped_layer_path"], cv2.IMREAD_UNCHANGED)
        
        # 2. PENYATUAN DASAR (Teks + Tekstur Kartu)
        assembled_ktp = overlay_transparent(template_img, warped_layer)
        
        if data["skenario"] == "A":
            # --- SKENARIO A: Selesai di sini ---
            final_out = assembled_ktp
            
        elif data["skenario"] == "B":
            # --- SKENARIO B: Penggantian Background ---
            mask_path = data["template"]["mask_path"]
            bg_gan_path = data["background_gan"]
            
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            bg_gan = cv2.imread(bg_gan_path)
            
            h_temp, w_temp = template_img.shape[:2]
            
            # Sesuaikan ukuran BG GAN agar seukuran template
            # (Crop bagian tengah agar rasio tekstur GAN tidak rusak)
            h_bg, w_bg = bg_gan.shape[:2]
            scale = max(w_temp/w_bg, h_temp/h_bg)
            new_w, new_h = int(w_bg * scale), int(h_bg * scale)
            bg_gan_resized = cv2.resize(bg_gan, (new_w, new_h))
            
            start_x = (new_w - w_temp) // 2
            start_y = (new_h - h_temp) // 2
            bg_gan_cropped = bg_gan_resized[start_y:start_y+h_temp, start_x:start_x+w_temp]
            
            # Step B1: Harmonisasi Warna KTP terhadap BG GAN
            harmonized_ktp = apply_reinhard_color_transfer(assembled_ktp, bg_gan_cropped, mask, transfer_ratio=0.25)
            
            # Step B2: Tambahkan Drop Shadow ke BG GAN
            bg_with_shadow = generate_drop_shadow(bg_gan_cropped, mask)
            
            # Step B3: Tempel KTP yang sudah diharmonisasi ke BG yang ada bayangannya
            final_out = np.zeros_like(bg_with_shadow)
            mask_alpha = mask / 255.0
            for c in range(3):
                final_out[:, :, c] = (harmonized_ktp[:, :, c] * mask_alpha + bg_with_shadow[:, :, c] * (1 - mask_alpha))
        
        # 3. SIMPAN HASIL
        out_path = os.path.join(OUTPUT_COMPOSITED_DIR, f"composited_{data['skenario']}_{i:03d}.jpg")
        cv2.imwrite(out_path, final_out)
        
        # Update path di JSON
        data["composited_path"] = out_path
        berhasil += 1
        
        if (i + 1) % 10 == 0:
            print(f"   Compositing {i + 1}/100 dataset selesai...")
            
    except Exception as e:
        print(f"❌ Error pada dataset ke-{i} (NIK: {data['nik']}): {e}")

# Simpan pembaruan JSON
with open(MAPPING_FILE, "w") as f:
    json.dump(final_datasets, f, indent=4)

print("="*50)
print(f"✅ FASE 6 SELESAI ({berhasil}/{len(final_datasets)} berhasil diproses)")
print(f"Waktu Eksekusi: {time.time() - start_time:.2f} detik.")
print(f"Silakan periksa folder '{OUTPUT_COMPOSITED_DIR}' untuk melihat hasilnya.")
print("="*50)

# Fase 7 - Finalization

In [ ]:

def apply_camera_physics(image):
    # 1. SLIGHT LENS BLUR (Fokus Lensa Sedikit Melunak)
    # Menggunakan kernel 3x3, sangat halus agar teks KTP tetap terbaca (OCR-friendly)
    blurred = cv2.GaussianBlur(image, (3, 3), 0)
    
    # 2. GLOBAL SENSOR NOISE (Bintik ISO Kamera)
    row, col, ch = blurred.shape
    mean = 0
    var = random.uniform(10, 25) # Variasi intensitas noise antar foto
    sigma = var ** 0.5
    
    # Generate matriks noise dan tambahkan ke gambar
    gauss_noise = np.random.normal(mean, sigma, (row, col, ch))
    gauss_noise = gauss_noise.reshape(row, col, ch)
    noisy_img = blurred + gauss_noise
    
    # Kliping nilai pixel agar tetap di rentang 0-255
    noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
    
    # 3. JPEG COMPRESSION ARTIFACTS
    # Mensimulasikan gambar yang dikirim via WhatsApp (kualitas 75 - 90)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), random.randint(75, 90)]
    result, encimg = cv2.imencode('.jpg', noisy_img, encode_param)
    final_img = cv2.imdecode(encimg, 1)
    
    return final_img

# ==========================================
# EKSEKUSI FINAL
# ==========================================
print(f"🚀 Memulai Fase 7: Finalisasi & Pembangkitan Ground Truth ML...")
start_time = time.time()
berhasil = 0

# List untuk menyimpan label ground truth untuk model ML-mu
ml_ground_truth = []

for i, data in enumerate(final_datasets):
    try:
        # Load gambar hasil compositing dari Fase 6
        composited_path = data.get("composited_path")
        if not composited_path or not os.path.exists(composited_path):
            print(f"⚠️ Melewati data ke-{i}: Gambar composited tidak ditemukan.")
            continue
            
        img = cv2.imread(composited_path)
        
        # Terapkan efek kamera
        final_output = apply_camera_physics(img)
        
        # Nama file final menggunakan NIK untuk mempermudah indexing
        nik_file = data.get('nik', f"UNKNOWN_{i}")
        final_filename = f"ktp_synthetic_{nik_file}.jpg"
        final_path = os.path.join(OUTPUT_FINAL_DIR, final_filename)
        
        # Simpan gambar
        cv2.imwrite(final_path, final_output)
        
        # ==========================================
        # UPDATE: GROUND TRUTH LENGKAP UNTUK ML
        # ==========================================
        clean_label = {
            "file_name": final_filename,
            "skenario_augmentasi": data.get("skenario", "B"),
            "data_teks": {
                "provinsi": data.get("provinsi", ""),
                "kota_kab": data.get("kota_kab", ""),
                "nik": data.get("nik", ""),
                "nama": data.get("nama", ""),
                "tempat_lahir": data.get("tempat_lahir", ""),
                "tgl_lahir": data.get("tgl_lahir", ""),
                "jenis_kelamin": data.get("jenis_kelamin", ""),
                "gol_darah": data.get("gol_darah", ""),
                "alamat": data.get("alamat", ""),
                "rt_rw": data.get("rt_rw", ""),
                "kel_desa": data.get("kel_desa", ""),
                "kecamatan": data.get("kecamatan", ""),
                "agama": data.get("agama", ""),
                "status_perkawinan": data.get("status_perkawinan", ""),
                "pekerjaan": data.get("pekerjaan", ""),
                "kewarganegaraan": data.get("kewarganegaraan", ""),
                "berlaku_hingga": data.get("berlaku_hingga", ""),
                "tgl_pembuatan": data.get("tgl_pembuatan", "")
            },
            "bounding_boxes": {
                "koordinat_kartu_ktp": data.get("template", {}).get("card_corners", []),
                "koordinat_pas_foto": data.get("template", {}).get("photo_corners", [])
            }
        }
        ml_ground_truth.append(clean_label)
        
        berhasil += 1
        if (i + 1) % 10 == 0:
            print(f"   Finalisasi {i + 1}/100 dataset selesai...")
            
    except Exception as e:
        print(f"❌ Error pada dataset ke-{i}: {e}")

# Simpan Ground Truth Super Lengkap
with open(GROUND_TRUTH_FILE, "w") as f:
    json.dump(ml_ground_truth, f, indent=4)

print("="*50)
print(f"🎉 SELURUH WORKFLOW SELESAI! 🎉")
print(f"Berhasil merender {berhasil} dataset KTP Sintetik yang siap di-training.")
print(f"Waktu Eksekusi: {time.time() - start_time:.2f} detik.")
print(f"\n📁 HASIL AKHIR:")
print(f"1. Gambar: Folder '{OUTPUT_FINAL_DIR}'")
print(f"2. Label ML Lengkap: File '{GROUND_TRUTH_FILE}'")
print("="*50)